# ENGRAM big model service (Qwen on Kaggle's GPU)

This notebook is **only part ⑤** of ENGRAM: it runs the big model as a web service and gives you a secure link.
Everything else (backend, small model, diary, web app) runs on your laptop and calls this link.

**Settings (right panel):** Accelerator **GPU T4** · Internet **On**. No dataset, no secrets.

**How to run:** **Run All** (an interactive session) and **keep this tab open** while you use the app.
After ~6 minutes the last cell prints two lines: copy them into `software/.env` on your laptop.
The link stops working when this session stops.

## 1. Get the code (from GitHub, ~1 min)

In [ ]:
MNEMO_REPO = 'https://github.com/slikhithreddy22/MNemo.git'          # the research code (model wrappers)
SOFTWARE_REPO = 'https://github.com/slikhithreddy22/engram-software.git'   # this software
import os, subprocess, sys, time, re, secrets, threading
os.chdir('/kaggle/working')
for url, name in ((MNEMO_REPO, 'mnemo'), (SOFTWARE_REPO, 'software')):
    if not os.path.exists(name):
        subprocess.run(['git', 'clone', '-q', '--depth', '1', url, name], check=True)
    print(name, subprocess.run(['git', '-C', name, 'log', '--oneline', '-1'], capture_output=True, text=True).stdout.strip())
!pip install -q "fastapi==0.120.0" "uvicorn==0.38.0" -U "transformers>=4.57" accelerate

## 2. Start the big model service (~5 min: Qwen3-VL-4B is 9 GB)

In [ ]:
os.chdir('/kaggle/working/software')
sys.path.insert(0, '/kaggle/working/software')
os.environ['MNEMO_PATH'] = '/kaggle/working/mnemo'
TOKEN = secrets.token_urlsafe(24)               # only someone with this token can use the service
os.environ['ENGRAM_VLM_TOKEN'] = TOKEN
import uvicorn
from services.vlm.app import app, describer
t = time.time()
describer()                                     # loads Qwen on the GPU
print(f'model loaded in {time.time() - t:.0f}s')
server = uvicorn.Server(uvicorn.Config(app, host='127.0.0.1', port=8002, log_level='warning'))
threading.Thread(target=server.run, daemon=True).start()
time.sleep(3)
import requests
print(requests.get('http://127.0.0.1:8002/health').json())

## 3. Secure public link (Cloudflare quick tunnel, no account needed)

In [ ]:
if not os.path.exists('/kaggle/working/cloudflared'):
    !wget -q -O /kaggle/working/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x /kaggle/working/cloudflared
tunnel = subprocess.Popen(['/kaggle/working/cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8002', '--no-autoupdate'],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
URL = None
for line in tunnel.stdout:
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m:
        URL = m.group(0)
        break
threading.Thread(target=lambda: [None for _ in tunnel.stdout], daemon=True).start()   # keep reading the log
time.sleep(5)
print(requests.get(URL + '/health', timeout=30).json())
print('\nCopy these two lines into software/.env on your laptop:\n')
print(f'ENGRAM_VLM_URL={URL}')
print(f'ENGRAM_VLM_TOKEN={TOKEN}')

## 4. Keep running
This cell runs until you stop it. Keep the tab open; the service stops when the session ends.

In [ ]:
while True:
    try:
        h = requests.get(URL + '/health', timeout=30).json()
        print(time.strftime('%H:%M'), 'service up:', h['model'], flush=True)
    except Exception as e:
        print(time.strftime('%H:%M'), 'service check failed:', e, flush=True)
    time.sleep(600)